<a href="https://colab.research.google.com/github/MOSAT-2026-SUMMER/CNN_Toy_Project/blob/main/src/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 9.9 MB/s eta 0:00:00


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

# ---------------------------------------------------------
# 1. 하이퍼파라미터 및 구글 드라이브 경로 설정
# ---------------------------------------------------------
# 첨부해주신 이미지의 바로가기 경로에 맞게 수정되었습니다.
FRAMES_DIR = "/content/drive/MyDrive/frames"
SAVE_DIR = "/content/drive/MyDrive/models"

# Colab VRAM 한계를 고려한 Batch Size
BATCH_SIZE = 4
EPOCHS = 20
LEARNING_RATE = 1e-4
MAX_SEQ_LENGTH = 100  # 5초 * 20FPS = 100프레임
HIDDEN_DIM = 64

# ---------------------------------------------------------
# 2. 데이터셋 클래스 정의
# ---------------------------------------------------------
class DrunkFrameDataset(Dataset):
    def __init__(self, root_dir, max_seq_length, transform=None):
        self.root_dir = root_dir
        self.max_seq_length = max_seq_length
        self.transform = transform
        self.video_folders = []
        self.labels = []

        # drunk: 1, sober: 0
        classes = {'sober': 0, 'drunk': 1}

        for class_name, label in classes.items():
            class_path = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_path):
                continue

            for video_folder in sorted(os.listdir(class_path)):
                folder_path = os.path.join(class_path, video_folder)
                if os.path.isdir(folder_path):
                    self.video_folders.append(folder_path)
                    self.labels.append(label)

    def __len__(self):
        return len(self.video_folders)

    def __getitem__(self, idx):
        folder_path = self.video_folders[idx]
        label = self.labels[idx]

        frame_paths = sorted(glob.glob(os.path.join(folder_path, "*.jpg")))

        if len(frame_paths) > self.max_seq_length:
            frame_paths = frame_paths[:self.max_seq_length]

        frames = []
        for img_path in frame_paths:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            frames.append(image)

        # 프레임 부족 시 Zero Padding
        while len(frames) < self.max_seq_length:
            frames.append(torch.zeros_like(frames[0]))

        # (Seq_Len, Channels, Height, Width)
        frames_tensor = torch.stack(frames)

        return frames_tensor, torch.tensor(label, dtype=torch.float32)

# ---------------------------------------------------------
# 3. 모델 아키텍처 (ResNet18 Fine-tuning + GRU)
# ---------------------------------------------------------
class ResNet18_GRU(nn.Module):
    def __init__(self, hidden_dim=64, num_gru_layers=3, dropout_rate=0.5):
        super(ResNet18_GRU, self).__init__()

        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # [Fine-tuning] layer4만 학습 허용, 나머지는 동결(Freeze)
        for name, param in resnet.named_parameters():
            if "layer4" in name:
                param.requires_grad = True
            else:
                param.requires_grad = False

        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
        resnet_out_dim = resnet.fc.in_features

        self.gru = nn.GRU(
            input_size=resnet_out_dim,
            hidden_size=hidden_dim,
            num_layers=num_gru_layers,
            batch_first=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        batch_size, seq_len, c, h, w = x.size()

        x = x.view(batch_size * seq_len, c, h, w)
        features = self.feature_extractor(x)
        features = features.view(batch_size, seq_len, -1)

        gru_out, _ = self.gru(features)
        last_time_step_out = gru_out[:, -1, :]

        output = self.classifier(last_time_step_out)
        return output.squeeze(-1)

# ---------------------------------------------------------
# 4. 학습 루프
# ---------------------------------------------------------
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"현재 사용 중인 디바이스: {device}")

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    dataset = DrunkFrameDataset(root_dir=FRAMES_DIR, max_seq_length=MAX_SEQ_LENGTH, transform=transform)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

    model = ResNet18_GRU().to(device)

    criterion = nn.BCELoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for batch_idx, (frames, labels) in enumerate(dataloader):
            frames, labels = frames.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(frames)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            predicted = (outputs > 0.5).float()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        epoch_loss = running_loss / len(dataloader)
        epoch_acc = correct / total if total > 0 else 0
        print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")

    print("학습이 완료되었습니다.")

    os.makedirs(SAVE_DIR, exist_ok=True)
    model_save_path = os.path.join(SAVE_DIR, "resnet18_gru_finetuned.pth")
    torch.save(model.state_dict(), model_save_path)
    print(f"모델이 구글 드라이브에 저장되었습니다: {model_save_path}")

if __name__ == "__main__":
    if os.path.exists(FRAMES_DIR):
        main()
    else:
        print(f"지정된 경로를 찾을 수 없습니다: {FRAMES_DIR}\n구글 드라이브 마운트가 정상적으로 되었는지 확인해주세요.")

현재 사용 중인 디바이스: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 113MB/s]


Epoch [1/20] Loss: 0.6921, Accuracy: 0.5333
Epoch [2/20] Loss: 0.6551, Accuracy: 0.5333
Epoch [3/20] Loss: 0.6369, Accuracy: 0.5333
Epoch [4/20] Loss: 0.5971, Accuracy: 0.6000
Epoch [5/20] Loss: 0.5875, Accuracy: 0.8000
Epoch [6/20] Loss: 0.5679, Accuracy: 0.9333
Epoch [7/20] Loss: 0.5763, Accuracy: 0.8000
Epoch [8/20] Loss: 0.5328, Accuracy: 0.9333
Epoch [9/20] Loss: 0.5082, Accuracy: 1.0000
Epoch [10/20] Loss: 0.4978, Accuracy: 1.0000
Epoch [11/20] Loss: 0.4910, Accuracy: 0.9333
Epoch [12/20] Loss: 0.4619, Accuracy: 0.9333
Epoch [13/20] Loss: 0.4531, Accuracy: 1.0000
Epoch [14/20] Loss: 0.4591, Accuracy: 1.0000
Epoch [15/20] Loss: 0.4393, Accuracy: 1.0000
Epoch [16/20] Loss: 0.4167, Accuracy: 1.0000
Epoch [17/20] Loss: 0.3785, Accuracy: 1.0000
Epoch [18/20] Loss: 0.3615, Accuracy: 1.0000
Epoch [19/20] Loss: 0.3668, Accuracy: 1.0000
Epoch [20/20] Loss: 0.3531, Accuracy: 1.0000
학습이 완료되었습니다.
모델이 구글 드라이브에 저장되었습니다: /content/drive/MyDrive/models/resnet18_gru_finetuned.pth
